# 3. Einstein time: 5-NN regression of $\log_{10} t_E$

In [ ]:
# --- Google Colab setup (does nothing when run locally) ---
import os, sys, subprocess

REPO_URL = "https://github.com/USERNAME/microlensing-dtw.git"  # <- edit to your repo
USE_DRIVE = True  # keep the repo (and results/, figures/) in Google Drive so they survive between notebooks

if "google.colab" in sys.modules:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        REPO_DIR = "/content/drive/MyDrive/microlensing-dtw"
    else:
        REPO_DIR = "/content/microlensing-dtw"
    # Clone only if the project isn't there yet (you can also upload the folder to Drive by hand)
    if not os.path.isdir(os.path.join(REPO_DIR, "mldtw")):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    # numpy, matplotlib, scikit-learn, joblib are preinstalled on Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tslearn", "VBMicrolensing"], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))

os.makedirs("../results", exist_ok=True)
os.makedirs("../figures", exist_ok=True)
print("Working directory:", os.getcwd())

In [ ]:
import sys
sys.path.insert(0, "..")
import json
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tslearn.neighbors import KNeighborsTimeSeriesRegressor
from mldtw.simulate import make_dataset, centre

In [ ]:
X_train, y_train, p_train = make_dataset(150, seed=1, classes=[2, 3])
X_test, y_test, p_test = make_dataset(50, seed=2, classes=[2, 3])
t_train = np.log10([p["tE"] for p in p_train])
t_test = np.log10([p["tE"] for p in p_test])
Xc_train, Xc_test = centre(X_train), centre(X_test)

In [ ]:
band = {"global_constraint": "sakoe_chiba", "sakoe_chiba_radius": 24}
metrics = {
    "Euclid": (False, "euclidean", {}),
    "Euclid, centred": (True, "euclidean", {}),
    "DTW": (False, "dtw", {}),
    "DTW band": (False, "dtw", band),
    "Centred + DTW": (True, "dtw", {}),
    "Centred + DTW band": (True, "dtw", band),
}
for g in [0.01, 0.1, 1, 10]:
    metrics[f"Centred soft-DTW, gamma={g}"] = (True, "softdtw", {"gamma": g})

In [ ]:
def predict(centred, metric, params):
    A, B = (Xc_train, Xc_test) if centred else (X_train, X_test)
    knn = KNeighborsTimeSeriesRegressor(n_neighbors=5, metric=metric, metric_params=params)
    knn.fit(A, t_train)
    return knn.predict(B)

pred = dict(zip(metrics, Parallel(n_jobs=len(metrics))(delayed(predict)(*m) for m in metrics.values())))
rmse = {n: float(np.sqrt(np.mean((p - t_test) ** 2))) for n, p in pred.items()}
json.dump(rmse, open("../results/tE_rmse.json", "w"), indent=1)
rmse

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, n in zip(axes, ["Euclid, centred", "Centred + DTW"]):
    ax.scatter(t_test, pred[n], c=y_test, s=12, cmap="coolwarm")
    ax.plot([-1.5, 2], [-1.5, 2], "k--")
    ax.set_title(f"{n}: {rmse[n]:.3f} dex")
    ax.set_xlabel("true $\\log_{10} t_E$ [d]")
axes[0].set_ylabel("predicted $\\log_{10} t_E$")
plt.tight_layout()
plt.savefig("../figures/tE_recovery.png", dpi=150)